# Stage1 公共分支重复收益输入：Qlib 训练结构预检

仅核对三对同义输入与固定训练中位数屏蔽的样本、标签、维度同序；不拟合或跑账户。

In [1]:
import torch
import hashlib,json,os,pathlib
import numpy as np,pandas as pd,qlib
from src.workflow.stage1_domain_heads import _read_source
from src.workflow.minute_pit_allocator import _primary_dataset,_auxiliary_dataset
from src.model.minute_pit_allocator import TARGETS,_flat_index
root=pathlib.Path.cwd();card=root/'research/stage1_common_return_dedup_preflight_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
assert sha(root/src['source_plan'])==src['source_plan_sha256']
assert sha(root/'research/stage1_domain_heads_20261018/summary.json')==src['source_summary_sha256']
for path,key in [('src/workflow/stage1_domain_heads.py','workflow_code_sha256'),('src/workflow/minute_pit_allocator.py','dataset_code_sha256'),('src/model/minute_pit_allocator.py','model_code_sha256')]:assert sha(root/path)==src[key]
parent=json.loads((root/src['source_plan']).read_text());source_spec=parent['sources']['training'];assert source_spec['recorder_id']==src['training_recorder_id'] and source_spec['archive']==src['training_NAS_sha256']
assert sha(root/src['alias_evidence'])
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/parent['provider_init']),region='us',kernels=1)
source,audit=_read_source(root,source_spec);first=source['models/base_outer_71']
features=source['inputs/features'];common=source['inputs/common'];frame=source['inputs/daily_frame'];timing=source['inputs/timing'];outer_aux=source['inputs/auxiliary']['outer']
assert features.index.equals(common.index) and features.index.equals(frame.index) and features.index.equals(timing.index)
assert len(first.base_features)==39 and len(common.columns)==16 and first.network.tail_encoder[0].in_features==16
aliases={'mom5':'soxx_ret5','mom20':'soxx_ret20','mom60':'soxx_ret60'}
differences={name:float(np.nanmax(np.abs(features[primary].to_numpy(float)-common[name].to_numpy(float)))) for name,primary in aliases.items()}
assert all(v<=1e-7 for v in differences.values())
inputs={'frame':frame,'features':features,'common':common,'timing':timing,'spec':first.spec}
train_end=str(first.train_end.date());bounds=source['inputs/plan']['outer_window']
base=_primary_dataset(inputs,train_end=train_end,predict_bounds=bounds);aux_base=_auxiliary_dataset(outer_aux,train_end)
train=_flat_index(base.prepare('train',col_set=['feature','label','common','meta']));maturity=pd.to_datetime(train[('meta','label_available_at')]);eligible=maturity.notna()&maturity.lt(first.cutoff)&(train.index<=first.train_end)&np.isfinite(train['label'][list(TARGETS)].to_numpy(float)).all(axis=1);actual=train.loc[eligible]
assert len(actual)==3014 and len(aux_base.prepare('train',col_set=['feature','label','meta']))==20741
medians={name:float(actual['common'][name].median()) for name in aliases};assert all(np.isfinite(v) for v in medians.values())
masked_common=common.copy(deep=True);masked_aux=outer_aux.copy(deep=True)
for name,value in medians.items():masked_common[name]=value;masked_aux[name]=value
candidate=_primary_dataset({**inputs,'common':masked_common},train_end=train_end,predict_bounds=bounds);aux_candidate=_auxiliary_dataset(masked_aux,train_end)
for segment in ('train','predict'):
 a=base.prepare(segment,col_set=['feature','label','common','meta']);b=candidate.prepare(segment,col_set=['feature','label','common','meta']);assert len(a)==len(b)
 for group in ('feature','label','meta'):pd.testing.assert_frame_equal(a[group],b[group],check_exact=True)
 rest=[x for x in a['common'].columns if x not in aliases];pd.testing.assert_frame_equal(a['common'][rest],b['common'][rest],check_exact=True)
 for name,value in medians.items():assert b['common'][name].eq(value).all()
a=aux_base.prepare('train',col_set=['feature','label','meta']);b=aux_candidate.prepare('train',col_set=['feature','label','meta']);assert len(a)==len(b)==20741
for group in ('label','meta'):pd.testing.assert_frame_equal(a[group],b[group],check_exact=True)
rest=[x for x in a['feature'].columns if x not in aliases];pd.testing.assert_frame_equal(a['feature'][rest],b['feature'][rest],check_exact=True)
for name,value in medians.items():assert b['feature'][name].eq(value).all()
result={'decision':'source_and_Qlib_DatasetH_common_return_alias_mask_feasible_no_fit','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['training_recorder_id'],'source_NAS_sha256':src['training_NAS_sha256'],'source_objects_read':len(source_spec['artifacts']),'full_source_rows':len(features),'original_primary_features':39,'original_common_features':16,'alias_max_raw_differences':differences,'train_mature_rows':len(actual),'auxiliary_rows':len(a),'prediction_rows':len(base.prepare('predict',col_set='feature')),'training_only_common_medians':medians,'unmasked_feature_label_meta_and_other_common_exact':True,'auxiliary_labels_meta_and_other_features_exact':True,'candidate_common_dimensions_unchanged':True,'SOXX_outcome_values_reported':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps(result,ensure_ascii=False,indent=2))


{
  "decision": "source_and_Qlib_DatasetH_common_return_alias_mask_feasible_no_fit",
  "plan_sha256": "79ef8d0288186be05743e356865f370815cb6aef4945e0d196d4b1c4595cfed0",
  "source_recorder_id": "68e31a0a0fad44e7ba1d1f7d856c7a0d",
  "source_NAS_sha256": "ee328cab292f178bf3582ef574fb411561c93b82545cd3bb90842ecd9a9a24f1",
  "source_objects_read": 10,
  "full_source_rows": 3966,
  "original_primary_features": 39,
  "original_common_features": 16,
  "alias_max_raw_differences": {
    "mom5": 5.960464477539063e-08,
    "mom20": 5.960464477539063e-08,
    "mom60": 5.960464477539063e-08
  },
  "train_mature_rows": 3014,
  "auxiliary_rows": 20741,
  "prediction_rows": 946,
  "training_only_common_medians": {
    "mom5": 0.006551146507263184,
    "mom20": 0.019736766815185547,
    "mom60": 0.05451840162277222
  },
  "unmasked_feature_label_meta_and_other_common_exact": true,
  "auxiliary_labels_meta_and_other_features_exact": true,
  "candidate_common_dimensions_unchanged": true,
  "SOXX_outcome

## 中文结论

原39项主输入与公共分支 mom5/20/60 三对同义差最大5.96e−8。训练3014、辅助20741、预测946行；训练中位数屏蔽只改变这3列，其他输入／标签／元数据完全一致，维度仍16。0拟合、0账户；下一步先做原版预测与账户复现门，再执行独立0bp去重账户。